![RMIT logo](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTlAppozhCGl50pjgEZu_tTmalWJi3TjpNwCg&s)
|Course|Foundations of Artificial Intelligence for STEM|
|-----|-------|
|Course Code|COSC2968/COSC3053|
|Lecturer(s)|Mr.Nhat-Quang Tran and Mrs.Anh Le Van|
|Full name - Student ID| Le Kim Quyen - s3983370<br>Ton Dat Toan - s4070286<br>Nguyen Hong Ha - s3979100<br>Huynh Bao Dang Khoa - s4045607<br>Nguyen Viet Ngan Anh - s4103086|

# Predicting the age of abalone

## 1. Introduction

*Write something here*

## 2. Data Exploration

### 2.1. Import and Functions
Installing and getting started with Terminal:
+ pip install matplotlib
+ pip install seaborn
+ pip install ydata-profiling
+ pip install ipywidgets

To install the necessary libraries such as matplotlib [ref], seaborn [ref] and ydata [ref].

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport

### 2.2. Load the dataset

In [ ]:
abalone_df = pd.read_csv('abalone.csv')
abalone_df.head() # Shows the first five samples

*Write something based on this format:*

**Catagorical Feature:**
+ `Type`: Type of abalones (M = male, F = female and I = infant)

**Numerical Features:**
+ `Length`: Longest shell measurement in mm (continuous)
+ `Diameter`: Perpendicular to length in mm (continuous)
+ `Height`: with meat in shell in mm (continuous)
+ `Whole weight`: whole abalone in g (continuous)
+ `Shucked weight`: weight of meat in g (continuous)
+ `Viscera weight`: gut weight (after bleeding) in g (continuous)
+ `Shell weight`: after being dried in g (continuous)
+ `Rings`: +1.5 gives the age in years (discrete)

In [ ]:
abalone_df.info() # Shows overview of each feature

*Write something? Q - idk*

### 2.3. Descriptive statistic, visualizations and correlation analysis

**An overview using ydata** [ref]**:**

In [ ]:
# An overview using ydata
ydata = ProfileReport(abalone_df, title = 'Abalone overview')
ydata.to_notebook_iframe()

*Write something? Q - idk*

**Scatter plot between every pair of feature using seaborn** [ref]**:**

In [ ]:
# An overview of the relationship between each pair feature
sns.set_theme() # set default theme for plot

# Plot features with each color for each type of abalone, including male (M), female (F) and infant (I)
abalone_pp = sns.pairplot(data = abalone_df, hue = 'Type', corner = True, palette = 'Set2');

*Write something? Q - idk*

**Compute correlations between features**

In [ ]:
# The variable contains correlations between features
abalone_corr = abalone_df.corr(numeric_only = True)
# abalone_corr

**Visualize using headmap from seaborn** [ref]**:**

In [ ]:
ones_corr = np.ones_like(abalone_corr, dtype = bool) # create a matrix contains boolean values with the same shape as abalone_corr
mask = np.triu(ones_corr) # a mask contains half triangle matrix of True and False

# Remove redundant the first row and the last column
adjusted_abalone_corr = abalone_corr.iloc[1:, :-1]
adjusted_mask_corr = mask[1:, :-1]

# Create plot
fig, ax = plt.subplots(figsize = (10, 8))

# Show heatmap
sns.heatmap(data = adjusted_abalone_corr, mask = adjusted_mask_corr, cmap = 'Blues',
            annot = True, fmt = '.2f', vmin = -1, vmax = 1,
            linecolor = 'white', linewidths = .5, annot_kws = {'fontsize': 14});

*Write something? Q - idk*

### 2.4. Discover the data to gain insights

**Distribution of Numerical feature values**

In [ ]:
abalone_df.describe()

*Write something? Q - idk*

**Distribution of Categorical feature values**

In [ ]:
abalone_df.describe(include = 'object')

*Write something? Q - idk*

**The relationship of diameter and height using jointplot from seaborn** [ref]**:**

In [ ]:
sns.jointplot(data = abalone_df, x = 'Diameter', y = 'Height', hue = 'Type');

*Write something? Q - idk*

**The relationship of diameter and height using FacetGrid from seaborn** [ref]**:**

In [ ]:
abalone_fg = sns.FacetGrid(data = abalone_df, col = 'Type')
abalone_fg.map(sns.scatterplot, 'Diameter', 'Height', alpha = .4);

*Write something? Q - idk*

**The relationship of diameter and rings using FaceGrid from seaborn** [ref]**:**

In [ ]:
abalone_fg = sns.FacetGrid(data = abalone_df, col = 'Type')
abalone_fg.map(sns.scatterplot, 'Diameter', 'Rings', alpha = .4);

*Write something? Q - idk*

## 3. Data Preprocessing

### 3.1. Handle missing values
There is no missing values to handle?? 😢

In [ ]:
# Calculate the percentage of missing values in each feature
for col in abalone_df.columns:
    missing_values = abalone_df[col].isna().sum() # Count the number of missing values
    print(f'Feature {col} has {missing_values / len(abalone_df) * 100}% missing values')

*Write something? Q - idk*

### 3.2. Identify and handle outliers
*Work here*

### 3.4. Separate labels from data
Choosing "Rings" column as the label.

In [ ]:
abalone_set_labels = abalone_df['Rings'].copy()
abalone_set = abalone_df.drop(columns = 'Rings')

# uncomment to see the output
# abalone_set.head(), abalone_set_labels.head()

### 3.4. Encode categorical features
There is one categorical feature in the dataset need to be encoded: `Type`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Create a class for encode columns
colTrans = ColumnTransformer(transformers = [('encode', OneHotEncoder(), [0])], remainder = 'passthrough')

# Encode
abalone_set = colTrans.fit_transform(abalone_set)
abalone_set, abalone_set.shape

In [ ]:
# Transform abalone_set_labels into the same format with abalone_set
abalone_set_labels = abalone_set_labels.values
abalone_set_labels, abalone_set_labels.shape

### 3.5. Split the dataset
*Work here*

In [ ]:
from sklearn.model_selection import train_test_split

# Split train_set, train_set_labels for training - test_set and test_set_labels for testing
train_set, test_set, train_set_labels, test_set_labels = train_test_split(abalone_set, abalone_set_labels, test_size = 0.2, random_state = 25)

### 3.6. Feature scaling
Scale features to zero mean and unit variance.

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()

train_set[:, 3:] = std_scaler.fit_transform(train_set[:, 3:]) # just only scale numerical features, not category feature
test_set[:, 3:] = std_scaler.transform(test_set[:, 3:])

## 4. Model Training and Evaluation

**Note:**
+ `train_set` contains set data for training
+ `train_set_labels` contains labels for training
+ `test_set` contains set data for testing
+ `test_set_labels` contains labels for testing

**Modules in lec:**
+ LinearRegression
+ DecisionTreeRegressor
+ RandomForestRegressor
+ PolynomialFeatures

## Linear Regression

In [19]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(train_set, train_set_labels)
print('\n____________ LinearRegression ____________')
print('Learned parameters: ', model.coef_, model.intercept_)

# 5.1.2 Compute R2 score and root mean squared error
def r2score_and_rmse(model, train_data, labels): 
    r2score = model.score(train_data, labels)
    from sklearn.metrics import mean_squared_error
    prediction = model.predict(train_data)
    mse = mean_squared_error(labels, prediction)
    rmse = np.sqrt(mse)
    return r2score, rmse      
r2score, rmse = r2score_and_rmse(model, train_set, train_set_labels)
print('\nR2 score (on training data, best=1):', r2score)
print("Root Mean Square Error: ", rmse.round(decimals=1))
        
# 5.1.3 Predict labels for some training instances
print("\nInput data: \n", train_set[0:9])
print("\nPredictions: ", model.predict(train_set[0:9]).round(decimals=1))
print("Labels:      ", list(train_set_labels[0:9]))


____________ LinearRegression ____________
Learned parameters:  [ 9.90672867e+11  9.90672867e+11  9.90672867e+11 -2.74536133e-01
  1.07836914e+00  1.01635742e+00  4.22222900e+00 -4.30212402e+00
 -1.21008301e+00  1.06311035e+00] -990672866899.2598

R2 score (on training data, best=1): 0.5469467929640909
Root Mean Square Error:  2.2

Input data: 
 [[ 0.          1.          0.         -1.66924878 -1.66137629 -1.37531483
  -1.40487461 -1.3282856  -1.38468733 -1.42038298]
 [ 1.          0.          0.          0.18744115  0.23512438  0.03688922
   0.02122828  0.35925722 -0.08120758 -0.2589068 ]
 [ 0.          1.          0.         -0.06011751 -0.01441519 -0.21987515
  -0.50567707 -0.6446954  -0.40821296 -0.30166052]
 [ 1.          0.          0.          0.31122048  0.58447976  0.55041797
   0.42708781  0.34346471  0.2140056   0.71017763]
 [ 0.          0.          1.          0.64129869  0.58447976  0.42203579
   0.89601323  1.21205292  0.60459535  0.66029828]
 [ 0.          0.         